# 03 Modeling: hyperparameter search, fitting, calibrators and operating points (Tables S3 to S5)

Saved searches, models, calibrators and thresholds in `models/` are reused when present; delete them to recompute.

In [ ]:
import os
import pandas as pd
import numpy as np
import utils

In [ ]:
train = utils.load_object(utils.DATA + "/train_2012_2014")
validation = utils.load_object(utils.DATA + "/validation_2015")
X_train = train[utils.FEATURES]
y_train = train["OBITO"]
X_train.shape, round(y_train.mean(), 4)

## Table S3: hyperparameter search space

In [ ]:
# the space searched for each gradient-boosting model (the logistic regression is not tuned)
search_space = pd.DataFrame([
    {"Model": "XGBoost", "Hyperparameter": "n_estimators", "Distribution": "choice", "Range or choices": "{100, 150, 200, 250, 300}"},
    {"Model": "XGBoost", "Hyperparameter": "max_depth", "Distribution": "choice", "Range or choices": "{3, 4, 5, 6, 7, 8}"},
    {"Model": "XGBoost", "Hyperparameter": "learning_rate", "Distribution": "uniform", "Range or choices": "0.01 to 0.3"},
    {"Model": "XGBoost", "Hyperparameter": "subsample", "Distribution": "uniform", "Range or choices": "0.5 to 1.0"},
    {"Model": "XGBoost", "Hyperparameter": "colsample_bytree", "Distribution": "uniform", "Range or choices": "0.5 to 1.0"},
    {"Model": "XGBoost", "Hyperparameter": "min_child_weight", "Distribution": "choice", "Range or choices": "{1, 3, 5, 10}"},
    {"Model": "XGBoost", "Hyperparameter": "reg_lambda", "Distribution": "uniform", "Range or choices": "0.0 to 5.0"},
    {"Model": "LightGBM", "Hyperparameter": "n_estimators", "Distribution": "choice", "Range or choices": "{100, 150, 200, 250, 300}"},
    {"Model": "LightGBM", "Hyperparameter": "num_leaves", "Distribution": "choice", "Range or choices": "{15, 31, 63, 127}"},
    {"Model": "LightGBM", "Hyperparameter": "learning_rate", "Distribution": "uniform", "Range or choices": "0.01 to 0.3"},
    {"Model": "LightGBM", "Hyperparameter": "min_child_samples", "Distribution": "choice", "Range or choices": "{20, 50, 100, 200}"},
    {"Model": "LightGBM", "Hyperparameter": "colsample_bytree", "Distribution": "uniform", "Range or choices": "0.5 to 1.0"},
    {"Model": "LightGBM", "Hyperparameter": "reg_lambda", "Distribution": "uniform", "Range or choices": "0.0 to 5.0"},
    {"Model": "CatBoost", "Hyperparameter": "iterations", "Distribution": "choice", "Range or choices": "{100, 150, 200, 250, 300, 1000}"},
    {"Model": "CatBoost", "Hyperparameter": "depth", "Distribution": "choice", "Range or choices": "{4, 5, 6, 7, 8}"},
    {"Model": "CatBoost", "Hyperparameter": "learning_rate", "Distribution": "uniform", "Range or choices": "0.01 to 0.3"},
    {"Model": "CatBoost", "Hyperparameter": "l2_leaf_reg", "Distribution": "quniform", "Range or choices": "1 to 10 step 1"},
    {"Model": "CatBoost", "Hyperparameter": "border_count", "Distribution": "choice", "Range or choices": "{32, 64, 128, 254}"},
])
utils.save_table(search_space, utils.SUPPLEMENT_TABLES, "Table_S3")
search_space

## Hyperparameter search (Hyperopt, 100 evaluations, 10-fold cross-validated F1)

In [ ]:
tuning = {}
for model_name in ["xgboost", "lightgbm", "catboost"]:
    best_path = utils.MODELS + "/best_params_" + model_name + ".pkl"
    trials_path = utils.MODELS + "/hyperopt_trials_" + model_name + ".pkl"
    if os.path.exists(best_path) and os.path.exists(trials_path):
        tuning[model_name] = {"best_parameters": utils.load_object(best_path),
                              "trials": utils.load_object(trials_path)}
        print("reused saved search:", model_name)
        continue
    result = utils.tune_hyperparameters(model_name, X_train, y_train,
                                        utils.CONTINUOUS, utils.CATEGORICAL, n_evaluations=100, n_folds=10)
    utils.save_object(result["best_parameters"], best_path)
    utils.save_object(result["trials"], trials_path)
    tuning[model_name] = result
    print("tuned:", model_name, "best F1:", round(result["trials"]["mean_cross_validated_f1"].max(), 4))

## Table S4: selected hyperparameter configurations

In [ ]:
best = {model_name: tuning[model_name]["best_parameters"] for model_name in ["xgboost", "lightgbm", "catboost"]}
table_s4 = pd.DataFrame({"XGBoost": best["xgboost"], "LightGBM": best["lightgbm"], "CatBoost": best["catboost"]})
table_s4 = table_s4.reset_index().rename(columns={"index": "Hyperparameter"}).sort_values("Hyperparameter")
utils.save_table(table_s4, utils.SUPPLEMENT_TABLES, "Table_S4")
table_s4

In [ ]:
# full search history (every evaluation of every model)
history = []
for model_name in ["xgboost", "lightgbm", "catboost"]:
    one = tuning[model_name]["trials"].copy()
    one.insert(0, "model", model_name)
    history.append(one)
search_history = pd.concat(history, ignore_index=True)
utils.save_table(search_history, utils.SUPPLEMENT_TABLES, "hyperparameter_search_history")
search_history.head(8)

## Fit the four final models on the training cohort

In [ ]:
final_params = dict(best)
final_params["logistic_regression"] = {}
for model_name, label in utils.MODELS_ORDER:
    model_path = utils.MODELS + "/model_" + model_name + ".pkl"
    if os.path.exists(model_path):
        print("reused saved model:", label)
        continue
    pipeline = utils.make_pipeline(model_name, final_params[model_name], utils.CONTINUOUS, utils.CATEGORICAL)
    pipeline.fit(X_train, y_train)
    utils.save_object(pipeline, model_path)
    print("fitted and saved:", label)

## Table S5: operating points and calibrators on the 2015 validation cohort

In [ ]:
X_val = validation[utils.FEATURES]
y_val = validation["OBITO"].values
operating_point_label = {"threshold 0.5": "Threshold 0.5", "max sens + spec": "Max sens + spec",
                         "fixed sensitivity 0.80": "Fixed sensitivity 0.80"}
metric_for_column = {"Sensitivity": "sensitivity", "Specificity": "specificity", "PPV": "ppv",
                     "NPV": "npv", "F1": "f1", "Balanced accuracy": "balanced_accuracy",
                     "Proportion flagged": "alert_rate"}

rows = []
for model_name, label in utils.MODELS_ORDER:
    model = utils.load_object(utils.MODELS + "/model_" + model_name)
    p_val = model.predict_proba(X_val)[:, 1]
    points = utils.operating_points(y_val, p_val)
    thresholds = dict(zip(points.operating_point, points.threshold))
    utils.save_object(thresholds, utils.MODELS + "/thresholds_" + model_name)
    utils.save_object({"sigmoid": utils.fit_sigmoid_calibrator(p_val, y_val),
                       "isotonic": utils.fit_isotonic_calibrator(p_val, y_val)},
                      utils.MODELS + "/calibrators_" + model_name)
    evaluation = utils.evaluate_with_confidence_intervals(y_val, p_val, thresholds)
    lookup = {row["metric"]: row for _, row in evaluation.iterrows()}
    for operating_point, threshold in thresholds.items():
        row = {"Model": label, "Operating point": operating_point_label[operating_point],
               "Threshold": round(threshold, 4)}
        for column, metric in metric_for_column.items():
            r = lookup[metric + " @ " + operating_point]
            row[column] = utils.estimate_with_ci(r["value"], r["ci_low"], r["ci_high"])
        rows.append(row)
table_s5 = pd.DataFrame(rows)
utils.save_table(table_s5, utils.SUPPLEMENT_TABLES, "Table_S5")
table_s5